# 06 - XGBoost (three tiers, three seeds)

**Runtime -> Run all. Resumes** - completed runs are detected by their saved
predictions and skipped. A GPU runtime helps but is not required (auto-detected).

Three tiers, each answering a different question:

* **Tier A - full corpus, lexical.** Large-scale DGA detection; comparable to
  the tier A baselines.
* **Tier B - probe universe, lexical + certificate.** What the certificate
  layer adds overall. Rides partly on certificate *presence* (83% of benign vs
  2.5% of malicious hold one) - real signal, but partly trivial.
* **Tier C - certificate-bearing domains only.** The hard question: among
  domains that all serve TLS, do certificate *properties* (free CA, age,
  validity, SAN patterns) separate live malicious infrastructure from benign?
  ~29.5k benign vs ~365 malicious - prevalence ~1.2%, finally realistic, so
  PR-AUC and FPR@95%TPR here mean what they say. This tier is the novel claim.

Gains over the notebook-05 baselines are attributable to what XGBoost actually
adds: native categorical handling (tld, issuer, key/sig algorithm), native
missing-value routing instead of sentinel filling, and boosting.

In [ ]:
# --- standard header ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'
if os.path.isdir(REPO):
    subprocess.run(['git','-C',REPO,'pull','-q'], check=False)
else:
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone','-q',f'https://{TOKEN}@{URL}',REPO], check=True)
sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
!pip -q install pyarrow zstandard xgboost

In [ ]:
import pandas as pd, numpy as np, xgboost as xgb
from pathlib import Path
from src.evaluate import splits, metrics, predictions
from src.utils import manifest as mf

DEVICE = 'cuda'
try:
    xgb.XGBClassifier(device='cuda', n_estimators=1).fit(
        np.zeros((4,2), dtype=np.float32), [0,1,0,1])
except Exception:
    DEVICE = 'cpu'
print('xgboost', xgb.__version__, '| device:', DEVICE)

FEATURES = Path(P['data']['features'])
X_all = pd.read_parquet(FEATURES/'fused_v1.parquet')
probe = set(pd.read_parquet(f"{P['data']['interim']}/probe_universe.parquet")['domain'])
print('full:', X_all.shape)

## Tiers and feature sets

In [ ]:
LEXICAL = ['length','core_length','n_labels','shannon_entropy','vowel_ratio',
           'digit_ratio','hyphen_count','max_consec_consonants','bigram_score',
           'trigram_score','unique_char_ratio','is_idn','has_digit','starts_with_digit']
CERT_NUM = ['has_certificate','is_free_ca','validity_days','days_until_expiry',
            'cert_age_days','is_expired','is_not_yet_valid','is_self_signed',
            'san_count','wildcard_san','cn_in_san','key_bits','short_validity',
            'very_fresh_cert','cn_san_mismatch','weak_key']
CATS_A = ['tld']
CATS_BC = ['tld','issuer_org','key_algorithm','sig_algorithm','san_bucket']

tierB_df = X_all[X_all['domain'].isin(probe)].copy()
tierC_df = X_all[X_all['has_certificate'] == True].copy()

TIERS = {
  'tierA_lexical': (X_all,    LEXICAL,                              CATS_A),
  'tierB_lexcert': (tierB_df, LEXICAL + CERT_NUM,                   CATS_BC),
  # cert presence is constant in tier C by construction - excluded
  'tierC_certonlypop': (tierC_df,
      LEXICAL + [c for c in CERT_NUM if c != 'has_certificate'],    CATS_BC),
}
for name, (d, num, cats) in TIERS.items():
    print(f'{name:18s} rows={len(d):>9,}  malicious={d["label"].mean():.4f}  '
          f'features={len(num)+len(cats)}')

In [ ]:
def matrix(df, num_cols, cat_cols, categories=None):
    """Build the design matrix.

    Category vocabularies are FIXED from the training part and applied to
    val/test: a value never seen in training becomes NaN, which XGBoost routes
    as missing. That is both required by XGBoost 3.x (categories must match
    across DMatrices) and methodologically right - an unseen TLD or issuer is
    genuinely unknown to the model, and must not be silently mapped to a
    training code.
    """
    Xm = df[num_cols].apply(pd.to_numeric, errors='coerce').astype(np.float32)
    for c in cat_cols:
        if categories is None:
            Xm[c] = df[c].astype('category')
        else:
            known = df[c].where(df[c].isin(categories[c]))   # unseen -> NaN
            Xm[c] = pd.Categorical(known, categories=categories[c])
    return Xm, df['label'].values, df['domain'].values

def train_categories(df, cat_cols):
    return {c: pd.Index(df[c].dropna().unique()) for c in cat_cols}

## Run grid

Early stopping on the frozen validation part of each split. Tier C keeps the
same frozen split membership - a cert-bearing domain stays in whichever part
the split assigned it - so no re-splitting is invented for the subpopulation.

In [ ]:
SPLITS = ['random_v1', 'family_disjoint_v1']
SEEDS  = [42, 43, 44]
PRED_DIR = Path(P['artifacts']['predictions'])
MODEL_DIR = Path(P['artifacts']['models'])

for tier, (data, num_cols, cat_cols) in TIERS.items():
    for split_name in SPLITS:
        sp = splits.load_split(P['data']['splits'], split_name)
        d = sp['domains']
        tr = data[data['domain'].isin(d['train'])]
        va = data[data['domain'].isin(d['val'])]
        te = data[data['domain'].isin(d['test'])]
        if tr['label'].nunique() < 2 or te['label'].nunique() < 2:
            print(f'SKIP {tier}/{split_name}: a part is single-class'); continue
        cats = train_categories(tr, cat_cols)
        Xtr, ytr, _   = matrix(tr, num_cols, cat_cols, cats)
        Xva, yva, _   = matrix(va, num_cols, cat_cols, cats)
        Xte, yte, dte = matrix(te, num_cols, cat_cols, cats)
        unseen = {c: int((~te[c].isin(cats[c]) & te[c].notna()).sum()) for c in cat_cols}
        print(f'{tier}/{split_name}: test values unseen in train -> {unseen}')
        spw = float((ytr==0).sum()) / max((ytr==1).sum(), 1)
        for seed in SEEDS:
            run_id = f'xgb_{tier}_{split_name}_s{seed}'
            if (PRED_DIR/f'{run_id}.parquet').exists():
                print('SKIP (done)', run_id); continue
            model = xgb.XGBClassifier(
                objective='binary:logistic', eval_metric='aucpr',
                tree_method='hist', device=DEVICE, enable_categorical=True,
                max_depth=8, learning_rate=0.05, n_estimators=2000,
                early_stopping_rounds=100, subsample=0.8, colsample_bytree=0.8,
                min_child_weight=5, reg_lambda=1.0,
                scale_pos_weight=spw, random_state=seed, verbosity=0)
            model.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
            scores = model.predict_proba(Xte)[:,1]
            m = metrics.evaluate(yte, scores)
            m['best_iteration'] = int(model.best_iteration)
            predictions.save(run_id, PRED_DIR, dte, yte, scores)
            model.save_model(str(MODEL_DIR/f'{run_id}.json'))
            mf.record(P['manifest'], run_id, f'xgb_{tier}',
                      {'tier': tier, 'num': num_cols, 'cat': cat_cols,
                       'scale_pos_weight': spw, 'device': DEVICE},
                      split_name, sp['split_file'], m, seed, repo_root=REPO)
            print(f'DONE {run_id:48s} roc={m["roc_auc"]:.4f} '
                  f'pr={m["pr_auc"]:.4f} fpr@95={m["fpr_at_95_tpr"]:.4f} '
                  f'iters={m["best_iteration"]}')
print('grid complete')

## Summary

In [ ]:
man = mf.load_manifest(P['manifest'])
runs = man[man['run_family'].str.startswith('xgb_')].copy()
runs['tier'] = runs['run_family'].str.replace('xgb_','')

agg = (runs.groupby(['tier','split_name'])
       [['metrics.roc_auc','metrics.pr_auc','metrics.fpr_at_95_tpr','metrics.mcc']]
       .agg(['mean','std']).round(4))
display(agg)
out = Path(P['results']['tables'])/'table_xgboost.csv'
agg.to_csv(out); print('wrote', out)

In [ ]:
# XGBoost vs the notebook-05 baselines, same tier and split (ROC, prevalence-free)
base = man[man['run_family'].str.startswith('baseline')].copy()
base['tier'] = base['run_id'].str.extract(r'(tier[AB]_\w+?)_')[0]
b = base.groupby(['tier','split_name'])['metrics.roc_auc'].mean()
x = runs.groupby(['tier','split_name'])['metrics.roc_auc'].mean()
cmp = pd.DataFrame({'best_baseline_roc': b, 'xgb_roc': x})
cmp['gain'] = (cmp['xgb_roc'] - cmp['best_baseline_roc']).round(4)
display(cmp.round(4))

In [ ]:
# Tier C is the headline for the certificate-properties claim: report it at
# its true prevalence with the metrics that matter there.
c = runs[runs['tier']=='tierC_certonlypop']
if len(c):
    display(c.groupby('split_name')
             [['metrics.pr_auc','metrics.roc_auc','metrics.fpr_at_95_tpr',
               'metrics.positive_rate']].agg(['mean','std']).round(4))
    print('Read: at ~1.2% prevalence, PR-AUC is the honest headline; the naive')
    print('predict-all-benign strategy scores PR-AUC ~= prevalence.')

---

**Next:** `07_cnn_bilstm` (character branch, tier A), then fusion. If tier C's
PR-AUC lands meaningfully above its ~0.012 prevalence floor, the
certificate-properties claim stands and fusion has both regimes to combine.